In [ ]:

!pip -q install -U datasets transformers peft accelerate bitsandbytes
!pip -q install evaluate rouge_score bert_score sentence-transformers nltk textstat

In [ ]:

import os
import json
import re
import gc
import numpy as np
import pandas as pd
import torch
import nltk

from google.colab import drive, userdata
drive.mount("/content/drive")

# Add HF_TOKEN in Colab Secrets (key icon) with notebook access enabled.
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Add your Hugging Face token as the Colab secret HF_TOKEN.")

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from tqdm.auto import tqdm

# NLTK resources
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/Clinical_NLP_Project"

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# 1K adapter only
ADAPTER_DIR = "/content/drive/MyDrive/llama32_1b_radiology_lora_adapter"

RESULTS_DIR = os.path.join(BASE_DIR, "results", "1K")
os.makedirs(RESULTS_DIR, exist_ok=True)

# New filenames: do not overwrite the original 1K, 5K or 10K results
PREDICTION_PATH = os.path.join(
    RESULTS_DIR,
    "1k_finetuned_predictions_shared100_regenerated.csv"
)

METRICS_CSV_PATH = os.path.join(
    RESULTS_DIR,
    "1k_finetuned_metrics_shared100_regenerated.csv"
)

METRICS_JSON_PATH = os.path.join(
    RESULTS_DIR,
    "1k_finetuned_metrics_shared100_regenerated.json"
)

assert os.path.isdir(ADAPTER_DIR), (
    f"1K adapter folder not found: {ADAPTER_DIR}\n"
    "Update ADAPTER_DIR to the exact folder containing the saved 1K adapter."
)

print("1K adapter:", ADAPTER_DIR)
print("Output folder:", RESULTS_DIR)

In [ ]:

DATASET_NAME = (
    "BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track"
)

dataset = load_dataset(DATASET_NAME)

shuffled = dataset["train"].shuffle(seed=42)

# This is the first 100 rows of the shared test range 10100:10600
test_dataset = shuffled.select(range(10100, 10200))

reports = [
    str(x or "")
    for x in test_dataset["radiology_report"]
]

references = [
    str(x or "")
    for x in test_dataset["layman_report"]
]

assert len(reports) == 100
assert len(references) == 100

print("Shared test samples:", len(reports))
print("First report:", reports[0][:250])

In [ ]:


compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model.eval()

print("1K adapter loaded.")

In [ ]:


def generate_summary(report):

    prompt = f"""
You are a medical communication assistant.

Convert the following radiology report into a simple,
patient-friendly explanation.

Do not add medical findings that are not present.
Do not add diagnoses, symptoms, treatment, causes,
recommendations, or prognosis.

Radiology Report:
{report}

Patient-friendly summary:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return generated_text

In [ ]:

predictions = []

for report in tqdm(reports, total=100, desc="Generating 1K predictions"):
    predictions.append(generate_summary(report))

assert len(predictions) == 100


predictions_df = pd.DataFrame({
    "radiology_report": reports,
    "reference_layman_summary": references,
    "generated_summary": predictions
})

predictions_df.to_csv(PREDICTION_PATH, index=False)

print("\nSaved predictions:", PREDICTION_PATH)
print("Rows saved:", len(predictions_df))

print("\nFIRST SAMPLE")
print("=" * 70)
print("REPORT:\n", reports[0])
print("\nREFERENCE:\n", references[0])
print("\nGENERATED:\n", predictions[0])

In [ ]:


from nltk.translate.bleu_score import corpus_bleu

references_tokenized = [
    [ref.lower().split()]
    for ref in references
]

predictions_tokenized = [
    pred.lower().split()
    for pred in predictions
]

bleu_score = corpus_bleu(
    references_tokenized,
    predictions_tokenized
)


from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for ref, pred in zip(references, predictions):
    scores = scorer.score(ref, pred)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

rouge1 = float(np.mean(rouge1_scores))
rouge2 = float(np.mean(rouge2_scores))
rougeL = float(np.mean(rougeL_scores))



from nltk.translate.meteor_score import meteor_score

meteor_scores = []

for ref, pred in zip(references, predictions):
    ref_tokens = nltk.word_tokenize(ref.lower())
    pred_tokens = nltk.word_tokenize(pred.lower())

    meteor_scores.append(
        meteor_score([ref_tokens], pred_tokens)
    )

meteor = float(np.mean(meteor_scores))

In [ ]:

from bert_score import score as bertscore

P, R, F1 = bertscore(
    predictions,
    references,
    model_type="roberta-large",
    lang="en",
    verbose=True
)

bert_precision = float(P.mean().item())
bert_recall = float(R.mean().item())
bert_f1 = float(F1.mean().item())


from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

semantic_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)

reference_embeddings = semantic_model.encode(
    references,
    batch_size=16,
    show_progress_bar=True
)

prediction_embeddings = semantic_model.encode(
    predictions,
    batch_size=16,
    show_progress_bar=True
)

semantic_scores = []

for ref_emb, pred_emb in zip(
    reference_embeddings,
    prediction_embeddings
):
    pair_score = cosine_similarity(
        [ref_emb],
        [pred_emb]
    )[0][0]

    semantic_scores.append(float(pair_score))

semantic_similarity = float(np.mean(semantic_scores))

In [ ]:


def count_syllables(word):
    word = word.lower()
    vowels = "aeiouy"
    count = 0
    previous_vowel = False

    for char in word:
        is_vowel = char in vowels

        if is_vowel and not previous_vowel:
            count += 1

        previous_vowel = is_vowel

    if word.endswith("e") and count > 1:
        count -= 1

    return max(1, count)


def readability_scores(text):
    words = re.findall(r"\b[a-zA-Z]+\b", text)
    sentences = re.split(r"[.!?]+", text)
    sentences = [s for s in sentences if s.strip()]

    if not words:
        return 0.0, 0.0, 0.0

    W = len(words)
    S = max(1, len(sentences))
    complex_words = sum(1 for word in words if len(word) >= 7)
    syllables = sum(count_syllables(word) for word in words)

    avg_words_sentence = W / S
    avg_syllables_word = syllables / W
    complex_word_percentage = complex_words / W * 100

    fkgl = (
        0.39 * avg_words_sentence
        + 11.8 * avg_syllables_word
        - 15.59
    )

    dale_chall = (
        0.1579 * complex_word_percentage
        + 0.0496 * avg_words_sentence
    )

    if complex_word_percentage > 5:
        dale_chall += 3.6365

    coleman_liau = (
        5.89 * (len(text.replace(" ", "")) / W)
        - 30.0 * (S / W)
        - 15.8
    )

    return fkgl, dale_chall, coleman_liau


readability = [
    readability_scores(text)
    for text in predictions
]

fkgl = float(np.mean([x[0] for x in readability]))
dale_chall = float(np.mean([x[1] for x in readability]))
coleman_liau = float(np.mean([x[2] for x in readability]))

In [ ]:

metrics = {
    "Evaluation Samples": 100,
    "BLEU": bleu_score,
    "ROUGE-1": rouge1,
    "ROUGE-2": rouge2,
    "ROUGE-L": rougeL,
    "METEOR": meteor,
    "BERTScore Precision": bert_precision,
    "BERTScore Recall": bert_recall,
    "BERTScore F1": bert_f1,
    "Semantic Similarity": semantic_similarity,
    "FKGL": fkgl,
    "Dale-Chall": dale_chall,
    "Coleman-Liau": coleman_liau
}

metrics_df = pd.DataFrame(
    list(metrics.items()),
    columns=["Metric", "1K Score"]
)

metrics_df.to_csv(METRICS_CSV_PATH, index=False)

with open(METRICS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

print("\n" + "=" * 70)
print("1K FINE-TUNED MODEL — SHARED 100-SAMPLE RESULTS")
print("=" * 70)

display(metrics_df)

print("\nSaved files:")
print("Predictions:", PREDICTION_PATH)
print("Metrics CSV:", METRICS_CSV_PATH)
print("Metrics JSON:", METRICS_JSON_PATH)



del model
del base_model
del semantic_model

gc.collect()
torch.cuda.empty_cache()

print("\nCompleted 1K prediction generation and metric evaluation.")

In [ ]:


from transformers import pipeline

nli_classifier = pipeline(
    "text-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=0 if torch.cuda.is_available() else -1
)

nli_labels = []

for report, generated in tqdm(
    zip(reports, predictions),
    total=len(reports),
    desc="NLI hallucination evaluation — 1K"
):
    result = nli_classifier({
        "text": report,
        "text_pair": generated
    })

    if isinstance(result, list):
        label = result[0]["label"].lower()
    else:
        label = result["label"].lower()

    nli_labels.append(label)

hallucinated_samples = sum(
    label != "entailment"
    for label in nli_labels
)

hallucination_percentage = (
    hallucinated_samples / len(nli_labels)
) * 100 if nli_labels else 0.0

print("1K NLI Hallucination Results")
print("=" * 40)
print("Total samples:", len(nli_labels))
print("Hallucinated Samples:", hallucinated_samples)
print(f"Hallucination Percentage: {hallucination_percentage:.2f}%")

# Add NLI results to the existing 1K metrics
metrics["NLI Entailment Samples"] = sum(
    label == "entailment" for label in nli_labels
)
metrics["NLI Non-Entailment Samples"] = hallucinated_samples
metrics["Hallucination Percentage"] = hallucination_percentage

# Save the updated metrics without changing 5K or 10K results
metrics_df = pd.DataFrame(
    list(metrics.items()),
    columns=["Metric", "1K Score"]
)

metrics_df.to_csv(METRICS_CSV_PATH, index=False)

with open(METRICS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

# Save individual NLI predictions for detailed analysis
nli_df = predictions_df.copy()
nli_df["nli_label"] = nli_labels
nli_df["nli_non_entailment"] = [
    label != "entailment" for label in nli_labels
]

nli_path = os.path.join(
    RESULTS_DIR,
    "1k_nli_hallucination_shared100.csv"
)

nli_df.to_csv(nli_path, index=False)

print("\nUpdated metrics:", METRICS_CSV_PATH)
print("Updated metrics JSON:", METRICS_JSON_PATH)
print("Per-sample NLI results:", nli_path)

display(metrics_df)